# Phase 4: Feature Normalization and Stronger Ranking Baselines

Objective: To evaluate whether feature normalization improves ranking robustness and early-rank performance.


### Phase 4 Goals:

Test whether normalization affects:
1. Aggregate metrics (P@K, NDCG@K, Failure@K)
2. Conditional failure rates (model-focused on feasible queries)
3. Avoidable failures (queries where relevant docs exist)
4. Score flatness and ties (error taxonomy component)
5. Generalization to MQ2008 

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ndcg_score  
import pickle
import warnings
warnings.filterwarnings('ignore')

#Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

#Setup paths
PROJECT_ROOT=Path.cwd().parent
DATA_RAW=PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED=PROJECT_ROOT / 'data' / 'processed'
PHASE3_OUTPUT=DATA_PROCESSED / 'phase3_baseline'
PHASE4_OUTPUT=DATA_PROCESSED / 'phase4_normalization'
PHASE4_OUTPUT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Phase 4 output directory: {PHASE4_OUTPUT}")

#Constants
K_VALUES=[1, 3, 5, 10]
ZERO_VAR_FEATURES=[f'f{i}' for i in range(6, 11)]  # f6-f10

Project root: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding
Phase 4 output directory: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase4_normalization


# 1. Data Loading

In [5]:
def parse_letor_file(filepath):
    """Parsing LETOR format file."""
    data=[]
    with open(filepath, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line=line.strip()
            if not line:
                continue
            try:
                parts=line.split('#')
                feature_part=parts[0].strip()
                docid=None
                if len(parts)>1:
                    comment=parts[1].strip()
                    if 'docid' in comment:
                        docid=comment.split('=')[1].strip()
                tokens=feature_part.split()
                label=int(tokens[0])
                qid=int(tokens[1].split(':')[1])
                features={}
                for token in tokens[2:]:
                    fid, value=token.split(':')
                    features[int(fid)]=float(value)
                data.append({
                    'label': label,
                    'qid': qid,
                    'features': features,
                    'docid': docid
                })
            except Exception as e:
                print(f"Error parsing line {line_num}: {e}")
                continue
    return data


def data_to_dataframe(data):
    """Converting parsed data to DataFrame."""
    records=[]
    for item in data:
        record={'label': item['label'], 'qid': item['qid'], 'docid': item['docid']}
        for fid, value in item['features'].items():
            record[f'f{fid}']=value
        records.append(record)
    return pd.DataFrame(records)


#Loading MQ2007 Fold1
print("Loading MQ2007 Fold1...")
train_df_2007=pd.read_csv(DATA_PROCESSED / 'fold1_train_processed.csv')

FOLD1_DIR_2007=DATA_RAW / 'MQ2007' / 'Fold1'
test_file_2007=FOLD1_DIR_2007 / 'test.txt'

test_data_2007=parse_letor_file(str(test_file_2007))
test_df_2007=data_to_dataframe(test_data_2007)

print(f"  MQ2007 Train: {train_df_2007.shape} ({train_df_2007['qid'].nunique()} queries)")
print(f"  MQ2007 Test: {test_df_2007.shape} ({test_df_2007['qid'].nunique()} queries)")
print(f"  Evaluating on: TEST SPLIT")

#Loading MQ2008 Fold1
print("\nLoading MQ2008 Fold1...")
FOLD1_DIR_2008=DATA_RAW / 'MQ2008' / 'Fold1'

if not FOLD1_DIR_2008.exists():
    raise FileNotFoundError(
        f"MQ2008 directory not found at: {FOLD1_DIR_2008}\n"
        f"Please download MQ2008 and extract to: {DATA_RAW / 'MQ2008'}\n"
        "Expected structure: data/raw/MQ2008/Fold1/train.txt and test.txt"
    )

train_file_2008=FOLD1_DIR_2008 / 'train.txt'
test_file_2008=FOLD1_DIR_2008 / 'test.txt'


if not (train_file_2008.exists() and test_file_2008.exists()):
    raise FileNotFoundError(f"MQ2008 train/test files not found in {FOLD1_DIR_2008}")

train_data_2008=parse_letor_file(str(train_file_2008))
test_data_2008=parse_letor_file(str(test_file_2008))

train_df_2008=data_to_dataframe(train_data_2008)
test_df_2008=data_to_dataframe(test_data_2008)

print(f"  MQ2008 Train: {train_df_2008.shape} ({train_df_2008['qid'].nunique()} queries)")
print(f"  MQ2008 Test: {test_df_2008.shape} ({test_df_2008['qid'].nunique()} queries)")

#Identifying features
feature_cols=[col for col in train_df_2007.columns if col.startswith('f') and col[1:].isdigit()]
feature_cols=sorted(feature_cols, key=lambda x: int(x[1:]))
active_features=[f for f in feature_cols if f not in ZERO_VAR_FEATURES]

print(f"\nFeatures:")
print(f"  Total: {len(feature_cols)}")
print(f"  Zero-variance excluded: {ZERO_VAR_FEATURES}")
print(f"  Active: {len(active_features)}")


#Ensuring all feature columns exist (f1..f46)
ALL_FEATURES=[f"f{i}" for i in range(1, 47)]

for df_name, df in [("test_df_2007", test_df_2007),
                    ("train_df_2008", train_df_2008),
                    ("test_df_2008", test_df_2008)]:
    missing=[c for c in ALL_FEATURES if c not in df.columns]
    if missing:
        for c in missing:
            df[c]=0.0
        print(f"Filled {len(missing)} missing feature columns in {df_name}: {missing[:5]}{'...' if len(missing)>5 else ''}")

#ensuring consistent column order
test_df_2007 = test_df_2007[['label','qid','docid'] + ALL_FEATURES]
train_df_2008 = train_df_2008[['label','qid','docid'] + ALL_FEATURES]
test_df_2008  = test_df_2008[['label','qid','docid'] + ALL_FEATURES]


Loading MQ2007 Fold1...
  MQ2007 Train: (42158, 49) (1017 queries)
  MQ2007 Test: (13652, 49) (336 queries)
  Evaluating on: TEST SPLIT

Loading MQ2008 Fold1...
  MQ2008 Train: (9630, 49) (471 queries)
  MQ2008 Test: (2874, 49) (156 queries)

Features:
  Total: 46
  Zero-variance excluded: ['f6', 'f7', 'f8', 'f9', 'f10']
  Active: 41


## 1.1 Verifying consistency with Phase 3

In [6]:
#Loading Phase 3 results for verification
phase3_query_metrics=pd.read_csv(PHASE3_OUTPUT / 'baseline_query_metrics.csv')

print("Phase 3 Baseline (for verification):")
print(f"  Test queries: {len(phase3_query_metrics)}")
print(f"  Failure@5 (primary): {phase3_query_metrics['Failure@5_primary'].mean():.4f}")
print(f"  Queries with relevant docs: {(phase3_query_metrics['num_relevant_1'] > 0).sum()}")

queries_with_rel=phase3_query_metrics[phase3_query_metrics['num_relevant_1'] > 0]
print(f"  Conditional Failure@5: {queries_with_rel['Failure@5_primary'].mean():.4f}")

print(f"\nPhase 4 will evaluate on the same test split to ensure comparability.")

Phase 3 Baseline (for verification):
  Test queries: 336
  Failure@5 (primary): 0.2679
  Queries with relevant docs: 290
  Conditional Failure@5: 0.1517

Phase 4 will evaluate on the same test split to ensure comparability.


# 2. Evaluation Functions (Phase 3 Validated)

In [ ]:
def precision_at_k(ranked_labels, k, relevance_threshold=1):
    """
    Computing Precision@K with validation.
    Uses denominator min(k, num_docs) as per Phase 3.
    """
    if relevance_threshold not in [1, 2]:
        raise ValueError(f"Invalid relevance_threshold: {relevance_threshold}")
    
    if k<=0:
        raise ValueError(f"Invalid k:{k}")
    
    if len(ranked_labels)==0:
        return 0.0
    
    k_actual=min(k, len(ranked_labels))
    top_k=ranked_labels[:k_actual]
    
    if relevance_threshold==1:
        relevant_count=sum(label>=1 for label in top_k)
    else:
        relevant_count=sum(label==2 for label in top_k)
    
    return relevant_count / k_actual


def is_failure_at_k(ranked_labels, k, relevance_threshold=1):
    """
    Determine if query is a failure at K with validation.
    """
    if relevance_threshold not in [1, 2]:
        raise ValueError(f"Invalid relevance_threshold: {relevance_threshold}")
    
    if k<=0:
        raise ValueError(f"Invalid k: {k}")
    
    if len(ranked_labels)==0:
        return True
    
    top_k_labels=ranked_labels[:k]
    
    if relevance_threshold==1:
        has_relevant=any(label>=1 for label in top_k_labels)
    else:
        has_relevant=any(label==2 for label in top_k_labels)
    
    return not has_relevant


def ndcg_at_k(ranked_labels, k):
    """
    Computing NDCG@K (gain-matched with sklearn).
    Verified in Phase 3.
    """
    if k<=0:
        raise ValueError(f"Invalid k:{k}")

    if len(ranked_labels)==0:
        return 0.0
    
    #Computing DCG@K
    dcg=0.0
    for i, label in enumerate(ranked_labels[:k]):
        dcg+=(2**label - 1) / np.log2(i + 2)
    
    #Computing IDCG@K
    ideal_labels=sorted(ranked_labels, reverse=True)[:k]
    idcg=0.0
    for i, label in enumerate(ideal_labels):
        idcg+=(2**label - 1) / np.log2(i + 2)
    
    if idcg==0:
        return 0.0
    
    return dcg / idcg


print("Evaluation functions defined (Phase 3 validated)")

Evaluation functions defined (Phase 3 validated)


In [ ]:
print("="*80)
print("NDCG VERIFICATION (Gain-Matched with sklearn)")
print("="*80)

#Sampling 5 queries from test set
sample_qids=test_df_2007['qid'].unique()[:5]
max_diff=0.0

print("\nVerifying NDCG computation on sample queries:")
for qid in sample_qids:
    query_data=test_df_2007[test_df_2007['qid']==qid]
    labels=query_data['label'].values
    if len(labels)==0:
        continue

    #Gain-matched: our ndcg uses (2^label-1)
    gains=(2**labels)-1

    y_score=np.arange(len(labels),0,-1)
    
    for k in [3, 5, 10]:
        #Our implementation
        our_ndcg=ndcg_at_k(labels, k)
        
        #sklearn implementation (reshape for API)
        if len(labels)>0:
            sklearn_ndcg=ndcg_score(
                [gains], 
                [y_score],  
                k=k
            )
            diff=abs(our_ndcg - sklearn_ndcg)
            max_diff=max(max_diff, diff)
            
            if diff>1e-9:
                print(f" qid={qid}, K={k}: ours={our_ndcg:.6f}, sklearn={sklearn_ndcg:.6f}, diff={diff:.2e}")

print(f"\nMaximum difference: {max_diff:.2e}")
if max_diff < 1e-9:
    print("NDCG implementation verified (gain-matched with sklearn)")
else:
    print(f"Warning: NDCG difference exceeds tolerance")

NDCG VERIFICATION (Gain-Matched with sklearn)

Verifying NDCG computation on sample queries:

Maximum difference: 8.33e-17
NDCG implementation verified (gain-matched with sklearn)


# 3. Normalization function

In [17]:
def normalize_global(train_features, test_features):    #computing statistics once on the training set, then applying them everywhere
    """
    Applying global StandardScaler normalization.
    
    Parameters:
    -----------
    train_features: array(n_train, n_features)
    test_features: array(n_test, n_features)
    
    Returns:
    --------
    train_norm, test_norm, scaler
    """
    scaler=StandardScaler()
    train_norm=scaler.fit_transform(train_features) #fit computes mean and std. transform applies the formula
    test_norm=scaler.transform(test_features)       #uses the mean and std from train_features to avoid leakage
    return train_norm, test_norm, scaler


#As phase 2 showed, there is a very high within-query variance
#So we are doing normalization per query below
def normalize_per_query(df, feature_cols, qid_col='qid'):
    """
    Applying per-query z-score normalization.
    Uses ddof=0 consistently (population std).
    Guards against zero-variance within query.
    
    Parameters:
    -----------
    df: DataFrame with qid and features
    feature_cols: list of feature column names
    
    Returns:
    --------
    df_normalized: DataFrame copy with normalized features
    """

    #Normalizes inside each query, independently
    #Each query gets its own mean and std
    df_norm=df.copy()
    
    for qid, group in df.groupby(qid_col):
        for feat in feature_cols:
            values=group[feat].values
            mean=values.mean()
            std=values.std(ddof=0)  #Population std (ddof=0)
            
            if std>1e-10:  #Guard against zero-variance
                normalized=(values-mean)/std
            else:
                normalized=values -mean  #Center only
            
            df_norm.loc[group.index, feat]=normalized
    
    return df_norm


print(" Normalization functions defined")
print("\nNormalization details:")
print("  Global: StandardScaler (z-score) fitted on train")
print("  Per-Query: Z-score within query, ddof=0, guards zero-variance")

 Normalization functions defined

Normalization details:
  Global: StandardScaler (z-score) fitted on train
  Per-Query: Z-score within query, ddof=0, guards zero-variance
